# OpenSeek Track 3 — Free Kaggle GPU runner (self-hosted Qwen3-4B)

Runs the full ICL annotation pipeline against a **self-hosted Qwen3-4B** served by vLLM **inside this Kaggle notebook**. Zero external API budget. Produces `submission.zip` for upload to flagos.io.

**Prerequisites**
1. Your `Opentract` repo is **public** on github.com. (The notebook clones it.)
2. Notebook settings (right sidebar):
   - **Accelerator = GPU T4 x2** (free) or **GPU P100** (free).
   - **Internet = on** (needed once to pull the model weights from HuggingFace).
3. **No API key needed.** No HuggingFace token needed — Qwen3-4B is publicly downloadable.

Total runtime budget on Kaggle: 12h sessions, 30h/week of free GPU. The full run takes ~1–3h.

---

In [ ]:
# 1. Configuration -- edit these for your fork.

REPO_URL    = 'https://github.com/Eienel/Opentract'
REPO_BRANCH = 'claude/flagos-hackathon-strategy-MeA51'

MODEL_NAME  = 'Qwen/Qwen3-4B'   # exact model required by the competition
VLLM_PORT   = 8000

# Knobs
CONCURRENCY  = 16      # safe for local vLLM; raise if GPU util is low
MAX_DEMO_TOK = 28000
STRATEGY     = 'first_n'
LIMIT_TESTS  = None    # e.g. 5 for a smoke run; None means full eval

In [ ]:
# 2. Sanity-check GPUs and pick a compatible dtype.
# T4/P100 are pre-Ampere (compute capability < 8.0) and do NOT support bfloat16 --
# vLLM will refuse to start with bf16 on them. We auto-detect and pick fp16 instead.

import subprocess
out = subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total,compute_cap', '--format=csv,noheader']).decode()
print(out)
lines = [l.strip() for l in out.strip().split('\n') if l.strip()]
n_gpus = len(lines)
assert n_gpus >= 1, 'No GPU detected. Set Accelerator to GPU T4 x2 (or P100) in the right sidebar and re-run.'
TP_SIZE = 2 if n_gpus >= 2 else 1

min_cc = min(float(l.split(',')[2].strip()) for l in lines)
DTYPE = 'bfloat16' if min_cc >= 8.0 else 'float16'
print(f'min compute_cap = {min_cc}  =>  dtype = {DTYPE}')
print(f'using tensor_parallel_size = {TP_SIZE}')

In [ ]:
# 3. Clone the repo and install pipeline deps.

import os, subprocess, sys

REPO_DIR = '/kaggle/working/Opentract'
if not os.path.exists(REPO_DIR):
    subprocess.check_call(['git', 'clone', '--depth', '1', '-b', REPO_BRANCH, REPO_URL, REPO_DIR])

os.chdir(REPO_DIR)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'])
print('repo + pipeline deps ready in', REPO_DIR)

In [ ]:
# 4. Fetch the 8 official competition datasets (~10 MB).

subprocess.check_call([sys.executable, 'openseek/scripts/fetch_data.py'])

In [ ]:
# 5. Install vLLM (~3-5 min). vLLM ships with the OpenAI-compatible server we need.

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'vllm'])
print('vllm installed')

In [ ]:
# 6. Start vLLM in the background, serving Qwen3-4B with prefix caching enabled.
#
# Prefix caching is CRITICAL: the per-task demo prefix (~28K tokens) is identical
# across hundreds of queries, so vLLM caches the prefill once and reuses it.
# This is the single biggest speedup for this workload.
#
# T4-friendly defaults: dtype auto-detected (fp16 on T4), --enforce-eager to skip
# Triton CUDA-graph capture (avoids known issues on Turing), XFORMERS attention
# backend (FlashAttention-2 needs Ampere+).

import subprocess, time, requests, os

VLLM_LOG = '/kaggle/working/vllm.log'

env = os.environ.copy()
if min_cc < 8.0:
    env['VLLM_ATTENTION_BACKEND'] = 'XFORMERS'

vllm_cmd = [
    sys.executable, '-m', 'vllm.entrypoints.openai.api_server',
    '--model', MODEL_NAME,
    '--host', '127.0.0.1',
    '--port', str(VLLM_PORT),
    '--tensor-parallel-size', str(TP_SIZE),
    '--max-model-len', '32768',
    '--gpu-memory-utilization', '0.90',
    '--dtype', DTYPE,
    '--enable-prefix-caching',
    '--enforce-eager',
]
print('launching:', ' '.join(vllm_cmd))
print('env overrides:', {k: env[k] for k in env if k.startswith('VLLM_')})

log_fh = open(VLLM_LOG, 'w')
vllm_proc = subprocess.Popen(vllm_cmd, stdout=log_fh, stderr=subprocess.STDOUT, env=env)

def _tail_log(from_offset):
    try:
        sz = os.path.getsize(VLLM_LOG)
        if sz > from_offset:
            with open(VLLM_LOG, 'r') as fh:
                fh.seek(from_offset)
                chunk = fh.read()
            print(chunk, end='')
            return sz
    except FileNotFoundError:
        pass
    return from_offset

# Poll until server is ready. First-run downloads the ~8GB model from HuggingFace,
# so allow up to ~15 minutes.
url = f'http://127.0.0.1:{VLLM_PORT}/v1/models'
deadline = time.time() + 900
last_log_size = 0
while time.time() < deadline:
    last_log_size = _tail_log(last_log_size)
    if vllm_proc.poll() is not None:
        # Process died -- print remaining log so we can debug
        _tail_log(last_log_size)
        with open(VLLM_LOG, 'r') as fh:
            tail = fh.read()[-6000:]
        raise RuntimeError(
            f'vLLM exited early (code {vllm_proc.returncode}).\n\n'
            f'=== last 6KB of {VLLM_LOG} ===\n{tail}'
        )
    try:
        r = requests.get(url, timeout=2)
        if r.status_code == 200:
            print('\nvLLM is ready:', r.json())
            break
    except Exception:
        pass
    time.sleep(5)
else:
    _tail_log(last_log_size)
    raise RuntimeError(f'vLLM did not become ready within 15 minutes. See {VLLM_LOG}')

In [ ]:
# 7. Point the pipeline at the local vLLM server.

os.environ['OPENAI_BASE_URL'] = f'http://127.0.0.1:{VLLM_PORT}/v1'
os.environ['OPENAI_MODEL']    = MODEL_NAME
os.environ['OPENAI_API_KEY']  = 'EMPTY'   # vLLM ignores auth by default
print('base_url =', os.environ['OPENAI_BASE_URL'])
print('model    =', os.environ['OPENAI_MODEL'])

In [ ]:
# 8. Connectivity smoke test -- one tiny call. Run BEFORE the long batch.

subprocess.check_call([sys.executable, 'openseek/scripts/check_endpoint.py'])

In [ ]:
# 9. Quick smoke run on a few samples per task (~5 min). Verifies the full path.

subprocess.check_call([
    sys.executable, 'openseek/scripts/run_all.py',
    '--strategy', STRATEGY,
    '--max-demo-tokens', str(MAX_DEMO_TOK),
    '--concurrency', str(CONCURRENCY),
    '--limit-tests', '5',
    '--out-dir', '/kaggle/working/run-smoke',
    '--submission', '/kaggle/working/submission-smoke.zip',
])

import json
for tid in range(1, 9):
    p = f'/kaggle/working/run-smoke/openseek-{tid}-v1.jsonl'
    with open(p) as fh:
        first = fh.readline().strip()
    print(f'task {tid}: {first[:120]}')

In [ ]:
# 10. FULL RUN -- all 8 tasks, all 3666 test samples. ~1-3 hours.

args = [
    sys.executable, 'openseek/scripts/run_all.py',
    '--strategy', STRATEGY,
    '--max-demo-tokens', str(MAX_DEMO_TOK),
    '--concurrency', str(CONCURRENCY),
    '--out-dir', '/kaggle/working/run-full',
    '--submission', '/kaggle/working/submission.zip',
]
if LIMIT_TESTS is not None:
    args += ['--limit-tests', str(LIMIT_TESTS)]

subprocess.check_call(args)

In [ ]:
# 11. Done. Inspect the submission ZIP, then download it from the Output panel
# (right sidebar) and upload on the flagos.io Submission tab.

import os, zipfile
sub = '/kaggle/working/submission.zip'
print(f'submission: {sub}  ({os.path.getsize(sub)/1024:.1f} KB)')
with zipfile.ZipFile(sub) as zf:
    for name in zf.namelist():
        info = zf.getinfo(name)
        with zf.open(name) as fh:
            n_lines = sum(1 for ln in fh if ln.strip())
        print(f'  {name}  {info.file_size} bytes  {n_lines} predictions')

In [ ]:
# 12. Cleanup -- stop the vLLM server (frees GPU for the next notebook session).

vllm_proc.terminate()
vllm_proc.wait(timeout=30)
print('vllm stopped')